<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg", align="left">
</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">Named Entity Recognition</p>
    <p style="margin: 0; text-align:right;">Màster universitari de Ciència de Dades (<i>Data Science</i>)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicació</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

#Reconeixement d'entitats anomenades (Named Entity Recognition)

**Autors:** Nadjet Bouayad-Agha & Josep Maria Sabater

**Objectiu Notebook**

Aquest Notebook mostra diferents models de *reconeixement d'entitats anomenades* en un text ('Named Entity Recognition' o 'NER' per les seves sigles en anglès), entenent com a 'entitat anomenada' una paraula o grup de paraules del text que fan referència a objectes del món real com ara persones, organitzacions o llocs.

<p>&nbsp</p>

**Organització del Notebook**

El Notebook s'estructura en les següents seccions:

* [Secció 1](#install) instal·la i importa les llibreries i models necessaris per a l'estudi. L'estudi es basarà en la llibreria spaCy i els models NER en català que aquesta llibreria proporciona.

* [Secció 2](#basic) mostra les classes i mètodes bàsics de spaCy necessaris per al treball.

* [Secció 3](#ner_outofthebox) experimenta amb les classes i atributs natius de spaCy relacionades amb la detecció d'entitats i els seus models NER en català.

* Les seccions [4](#ner_pattern) i [5](#ner_training) mostren com afegir nous tipus d'entitat a un model preexistent, bé a partir de patrons bé basant-se en entrenament de models.

* [Secció 6](#avaluacio) mostra com avaluar un model NER així com les magnituds d'avaluació NER usuals.



# <a name="install"></a>1 Importar llibreries i models

## 1.1 Instal·lar spaCy i altres llibreries

Es força la instal·lació de la versió spaCy 3.2.0 ('Google Colab' proporciona per defecte la versió 2.2.4 a la data de finalització d'aquest notebook, desembre 2021)

Nota: Consultar [aquí](https://spacy.io/usage#changelog) les diferents versions de spaCy.

In [ ]:
!pip install spacy==3.2.0

     |████████████████████████████████| 6.0 MB 8.0 MB/s 
     |████████████████████████████████| 628 kB 66.6 MB/s 
     |████████████████████████████████| 42 kB 1.3 MB/s 
     |████████████████████████████████| 181 kB 77.0 MB/s 
     |████████████████████████████████| 10.1 MB 69.1 MB/s 
     |████████████████████████████████| 451 kB 70.4 MB/s 
  Attempting uninstall: catalogue
    Found existing installation: catalogue 1.0.0
    Uninstalling catalogue-1.0.0:
      Successfully uninstalled catalogue-1.0.0
  Attempting uninstall: srsly
    Found existing installation: srsly 1.0.5
    Uninstalling srsly-1.0.5:
      Successfully uninstalled srsly-1.0.5
  Attempting uninstall: thinc
    Found existing installation: thinc 7.4.0
    Uninstalling thinc-7.4.0:
      Successfully uninstalled thinc-7.4.0
  Attempting uninstall: spacy
    Found existing installation: spacy 2.2.4
    Uninstalling spacy-2.2.4:
      Successfully uninstalled spacy-2.2.4


In [ ]:
# Connectar amb Google Drive. Contindrà models i fitxers necessaris per a aquest notebook
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

Mounted at /gdrive


In [ ]:
# Llibreria per a avaluar NER
!pip install nervaluate

## 1.2 Instal·lar models spaCy per a català 

Importació dels models spaCy que s'utilitzen en aquest notebook.

La pàgina https://spacy.io/models/ca conté la descripció dels models en català incorporats a Spacy, les tasques NLP en què han estat entrenats i l'esquema d'etiquetes de cada tasca. Aquests models són:

**Models nadius spaCy**

Basats en una xarxa CNN més capa CRF

* `ca_core_news_sm`  
* `ca_core_news_md`
* `ca_core_news_lg`

El sufix final 'sm', 'md' o 'lg' fa referència a la mida del model (https://github.com/explosion/spacy-models)

**Model Transformer**

Basat en model BERTa (RoBERTa-based Catalan language model) [1]

* `ca_core_news_trf`

<p>&nbsp</p>
[1] Armengol-Estapé, J. et al. (2021). Are Multilingual Models the Best Choice for Moderately Under-resourced Languages? A Comprehensive Assessment for Catalan. In Findings of the Association for Computational Linguistics: ACL-IJCNLP 2021 (pp. 4933–4946). Association for Computational Linguistics. (https://aclanthology.org/2021.findings-acl.437/)   
<p>&nbsp</p>




Es força la instal·lació de versions dels models spaCy per a català compatibles amb la versió spaCy 3.2.0. 

Nota: La compatibilitat es pot obtenir de la descripció [spaCy dels models en català](https://spacy.io/models/ca), opció 'release details' de cada model. La llista completa de compatibilitats es pot veure [aquí](https://github.com/explosion/spacy-models/blob/master/compatibility.json)

In [ ]:
!python -m spacy download ca_core_news_md-3.2.0 --direct

     |████████████████████████████████| 50.3 MB 1.9 MB/s 
✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_md')


In [ ]:
!python -m spacy download ca_core_news_trf-3.2.0 --direct

     |█████████████████████▋          | 312.4 MB 1.4 MB/s eta 0:01:48

In [ ]:
# Validar la compatibilitat de les versions
!python -m spacy validate

## 1.3 Importar spaCy i carregar models en català

Importar spaCy i carregar [models spaCy en català](https://spacy.io/models/ca) que s'utilitzaran en apartats següents.

In [ ]:
import spacy
print (f"Spacy version installed: {spacy.__version__}")

Spacy version installed: 2.2.4


Es disposa de 4 [models spaCy disponibles per a català](https://spacy.io/models/ca) [2].

Els tres primers models (`ca_core_news_sm`, `ca_core_news_md`, `ca_core_news_lg`) es basen en una xarxa neuronal pròpia de spaCy que consta d'una xarxa convolucional (CNN) i una capa CRF (conditional random fields).
El quart model, `ca_core_news_trf`, es basa en un [model transformer ROBERTa](https://huggingface.co/BSC-TeMU/roberta-base-ca)
<p>&nbsp</p>
[2] A desembre 2021
<p>&nbsp</p>

In [ ]:
# Load "ca_core_news_md" model
MODEL = "ca_core_news_md"
nlp_md = spacy.load(MODEL)
print (f"Model '{MODEL}' loaded correctly!")

Model 'ca_core_news_md' loaded correctly!


In [ ]:
# Load "ca_core_news_trf" model
MODEL = "ca_core_news_trf"
nlp_trf = spacy.load(MODEL)
print (f"Model '{MODEL}' loaded correctly!")

Model 'ca_core_news_trf' loaded correctly!


In [ ]:
# Altres llibreries i classes
import pandas as pd                     # Per a formatar resultats d'avaluació i tractament de fitxer anotat manualment

from spacy import displacy              # Per a imprimir el text i les seves entitats
from spacy.training import Example      # Per a crear objecte spaCy 'Example' per a avaluar textos amb spaCy
from spacy.scorer import Scorer         # Per a avaluar models amb spaCy
from nervaluate import Evaluator        # Per a avaluar models amb la llibreria nervaluate

import json                             # Per a recuperar fitxer d'anotacions realitzades per l'anotador extern https://www.m47.ai/ca/
from pathlib import Path                


Nota: Perquè sigui accessible la carpeta de recursos d'aquest notebook, per als no propietaris,  cal anar primer a carpetes compartides de Google Drive, a sobre de la carpeta "recursos" dins "ner_nel_material", prémer el botó dret i escollir “Afegeix una drecera a Drive” ("add shortcut to drive"). Després, es considerarà aquest directori com el directori actual:

In [ ]:
# Directori de treball de models i fitxers necessaris per a aquest notebook
%cd /gdrive/My Drive/ner_nel_material/resources
print (f"Current working directory: {Path.cwd()}")

/gdrive/My Drive/ner_nel_material/resources
Current working directory: /gdrive/My Drive/ner_nel_material/resources


# <a name="basic"></a>2 Mètodes bàsics spaCy

Aquest apartat introductori mostra els principals objectes i mètodes spaCy necessaris per a l'estudi NER que es realitzarà.




In [ ]:
def get_text_to_print(text):
  """Format given text.

    Parameters:
      text (str): text to print

    Returns:
      str: text formatted in 100 character lines with an initial line numbering the characters
  """
  line_length = 100
  line_poss   = "     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100"
  text        = text.replace("\n", " ")     # Perquè el caràcter salt de línia no salti a l'imprimir text formatat
  text        = text.replace("\r", " ")     # En textos procedents de Viquipèdia s'ha detectat caràcter '\r' que s'interpretaria com posicionar-se a l'inici de línia si no es canvia a blanc
  text_format = "\n".join([ f"{i//line_length:<5}{text[i:i+line_length]}"  for i in range(0, len(text), line_length) ])
  return line_poss + "\n" + text_format + "\n" + line_poss

In [ ]:
def get_tokens_to_print(model, text):
  """Print tokens of the text and its relevant attributes.

    Parameters:
      model (spaCy model): spaCy model used for tokenization
      text (str):  text to transform in a spaCy doc class.

    Returns: ---
  """
  doc = model(text)
  print (f"The text:\n\n{get_text_to_print(text)}\n\nwas converted in a spaCy object: {type(doc)}\n")
  print (f"Token-based analysis. Each token is a spaCy object: {type(doc[0])}\n")
  
  # Obtenir files per imprimir: capçaleres i files de contingut
  rows  = []
  # head_align: Llista de tuples. Cada tupla: capçalera de columna i la seva alineació a l'imprimir
  head_align  = [('Token', '<'), ('Lemma', '<'), ('Syntactic parent', '<'), ('#Tok', '>'), ('Chr_Start', '>'), ('Chr_End', '>'), ('POS', '<'), 
                 ('TAG', '<'), ('TAG meaning:', '<'), ('ENT', '<'), ('DEP', '<'), ('DEP meaning:', '<')]   
  head, align = list(zip(*head_align))  
  rows.append(head)                           # Capçalera
  rows.append(['='*len(i) for i in head])     # Subratllat capçalera
  for tok in doc:
    rows.append([tok.text, tok.lemma_, tok.head.text, str(tok.i), str(tok.idx), str(tok.idx+len(tok)-1), tok.pos_, 
                 tok.tag_, str(spacy.explain(tok.tag_))[:20], tok.ent_type_, tok.dep_, str(spacy.explain(tok.dep_))[:20]])
  
  # Amplada de cada columna: l'amplada de l'element més ample de la columna.
  columns       = zip(*rows)     # generador, cada element és una columna amb format tupla amb els valors de columna
  column_widths = [max(len(i) for i in col) for col in columns]

  # Imprimir per files amb alineació i amplada màxima per columna
  for row in rows:
    print(*[f"{row[i]:{align[i]}{column_widths[i]}}  " for i in range(0, len(row))])

## 2.1 spaCy pipeline

**spaCy** transforma el text a analitzar en una classe 'doc' spaCy ('spacy.tokens.doc.Doc'), els mètodes de la qual permetran realitzar tractaments NLP (https://spacy.io/usage/spacy-101#annotations).

La transformació es realitza a partir del model preentrenat carregat i el text objecte d'estudi. **spaCy** aplica els diferents components que el model té activats:

![pipeline_spacy](https://spacy.io/pipeline-fde48da9b43661abcdf62ab70a546d71.svg)
<p>&nbsp</p>
(font: https://spacy.io/usage/spacy-101#architecture-pipeline) 

El model carregat **'ca_core_news_md'** conté els components:

In [ ]:
nlp_md.pipeline    # també nlp_md.pipe_names


[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7f872d4c29f0>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x7f872d4c2670>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7f872d32f9d0>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x7f872d217f50>),
 ('lemmatizer',
  <spacy.lang.ca.lemmatizer.CatalanLemmatizer at 0x7f872d26bcd0>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7f872d32f950>)]

S'observa, per tant, que els components del model en català 'nlp_md' són:  

* **tok2vec**. Segmenta el text en unitats elementals 'token'
* **morphologizer**. Característiques morfològiques dels token
* **Parser**. Prediu les dependències sintàctiques entre tokens 
* **attribute_ruler**. Proporciona atributs del token quan s'utilitzen patrons o regles.
* **lemmatizer**. Determina el lema de les paraules.
* **ner**. Detecta entitats i qualifica el tipus al qual pertanyen


In [ ]:
nlp_md.pipe_names

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

El model carregat de spaCy transforma un text en un objecte 'doc' de spaCy. A partir de l'objecte 'doc' s'accedeix a cadascun dels tokens (objecte `spacy.tokens.token.Token`) i als seus diferents atributs:

In [ ]:
# Convertir el text en objecte spaCy 'doc'
text = """Microsoft va ser fundada per Paul Allen i B. Gates el 4 d'abril de 1975 a Califòrnia per desenvolupar i \
comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat en el processador Intel 8080."""

get_tokens_to_print(nlp_md, text)    

The text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    Microsoft va ser fundada per Paul Allen i B. Gates el 4 d'abril de 1975 a Califòrnia per desenvolupa
1    r i comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat 
2    en el processador Intel 8080.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

was converted in a spaCy object: <class 'spacy.tokens.doc.Doc'>

Token-based analysis. Each token is a spaCy object: <class 'spacy.tokens.token.Token'>

Token            Lemma            Syntactic parent   #Tok   Chr_Start   Chr_End   POS     TAG     TAG meaning:           ENT    DEP        DEP meaning:          
=====            =====            ================   ====   =========   =======   ===     ===     ============           ===    ===        ============          
Microsoft        Microsoft        fundada           

## <a name="exercicis_2_2"></a>2.2 Exercicis

1. Modificar el text anterior i comprovar si la tokenització o les entitats són correctes, per exemple:

  - En comptes de la data "4 d'abril de 1975", provar amb "04/04/1975", i després amb "04-04-1975".
  - Substituir "Paul Allen" per "P. Allen" i per "P.Allen" (sense espai entre el punt i l'A).
  - Afegir "Ltd." després de "Microsoft", així "Microsoft Ltd. va ser fundada...".
  - Substituir "Microsoft" per una empresa desconeguda com "Super Soft", "Supersoft", "Macrosoft" i per  errors ortogràfics "Micro soft" i "Microoft".
  - Canviar "Intel 8080" per "Intel-8080".

2. Es poden desactivar els components del pipeline amb el mètode `disable_pipe`. Provar d'afegir:

  `nlp_md.disable_pipe("ner")`

  i/o

  `nlp_md.disable_pipe("parser")`

  abans d'analitzar el text i veure quina informació desapareix. Quin pot ser el motiu per desactivar un component?

3. Provar d'analitzar el text amb el model amb transformers (nlp_trf), quines diferències es poden observar pel que fa a les entitats anomenades?

# <a name="ner_outofthebox"></a>3 spaCy NER. Out-of-the-box.

Els 4 [models spaCy per a català](https://spacy.io/models/ca) ja han estat entrenats per a realitzar reconeixement d'entitats. 



## 3.1 Model `ca_core_news_md`. Obtenció d'entitats. El tipus Span.
Obtenció de les entitats utilitzant el model ja carregat `ca_core_news_md` i els mètodes natius de la classe 'doc' de spaCy. 

D'acord amb [l'esquema d'etiquetes del model](https://spacy.io/models/ca#ca_core_news_md) (apartat 'Label Scheme' i 'NER'), aquest reconeix les entitats "LOC" (noms de lloc), "PER" (noms de persona), "ORG" (noms d'organització) i "MISC" (altres entitats anomenades que no són cap de les anteriors).

In [ ]:
def get_ents_to_print (model, text):
  """Print text and its entities.

    Parameters:
      model (model spaCy): spaCy model used for entity recognition
      text (str): text for entity recogniton.

    Returns: 
      doc (spaCy 'doc' class): doc object from text
  """
  doc = model (text)
  print (f"\nModel '{model.meta['lang']+'_'+model.meta['name']}' applied to text:\n\n{get_text_to_print(text)}\n\nhas detected the entities:\n")
  
  tokens_list = list(doc)    # Cada element de la llista és un objecte Token de spaCy
  
  print (f"{'Entity (ent.text)':<30}  Type  Tok_Start  Tok_End  Chr_Start  Chr_End  {'Entities (text string)':<30}  {'Entities (list Token)'}  ")
  print (f"{'=================':<30}  ====  =========  =======  =========  =======  {'======================':<30}  {'====================='}   ")
  for ent in doc.ents:
      print (f"{ent.text:<30}  {ent.label_:<4}  {ent.start:>9}  {ent.end:>7}  {ent.start_char:>9}  {ent.end_char:>7}  {text[ent.start_char:ent.end_char]:<30}  {tokens_list[ent.start:ent.end]}")
  print ("\n") 
  return doc   


In [ ]:
# Obtenció de les entitats i principals atributs associats d'un text.
text = """Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril de 1975 a Califòrnia per desenvolupar i \
comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat en el processador Intel 8080."""
doc = get_ents_to_print(nlp_md, text)


Model 'ca_core_news_md' applied to text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril de 1975 a Califòrnia per desenvolu
1    par i comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basa
2    t en el processador Intel 8080.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

has detected the entities:

Entity (ent.text)               Type  Tok_Start  Tok_End  Chr_Start  Chr_End  Entities (text string)          Entities (list Token)  
=================               ====  =========  =======  =========  =======  ======================          =====================   
Microsoft                       ORG           0        1          0        9  Microsoft                       [Microsoft]
Paul Allen                      PER           5        7         29       39

Notar que el mètode, per a l'exemple particular proposat: 

* Cataloga el token 'BASIC' com una entitat de tipus 'ORG'.    

<p>&nbsp</p>  

Així mateix, observar dels resultats anteriors:

* Una entitat és un objecte spaCy 'span'; similar en molts aspectes a la classe 'doc'. 
* Una entitat pot ser multitoken.
* Un token a la vegada és un objecte spaCy token
* Es pot obtenir una anàlisi dels tokens d'un span de manera similar a com s'ha fet per a la classe 'doc'.

<p>&nbsp</p> 

Així, per a l'entitat 'Paul Allen' de l'exemple anterior:

In [ ]:
# Exemple d'entitat multitoken
ent = doc.ents[1]
print (f"\nEntity: '{ent}':\n  It's a class: {type(ent)}.\n  Entity, as a Python string: '{ent.text}'.\n  Entity, as a list of tokens: {[tok for tok in ent]} ")
tok = ent[0]
print (f"\nFirst token of entity: '{ent}'\n  It's the token: '{tok}'.\n  It's a class: '{type(tok)}'.\n  Token, as a Python string: '{tok.text}'\n") 


Entity: 'Paul Allen':
  It's a class: <class 'spacy.tokens.span.Span'>.
  Entity, as a Python string: 'Paul Allen'.
  Entity, as a list of tokens: [Paul, Allen] 

First token of entity: 'Paul Allen'
  It's the token: 'Paul'.
  It's a class: '<class 'spacy.tokens.token.Token'>'.
  Token, as a Python string: 'Paul'



In [ ]:
# Anàlisi dels tokens d'una entitat, classe, spaCy 'span', similar a com s'ha realitzat per a la classe 'doc'
print (f"\nToken-based analysis of entity: '{doc.ents[1]}':\n")
print (f"{'Token':<15}  {'Lemma':<15}  {'Syntactic parent':<16}  #Tok  Chr_Start  Chr_End  POS    TAG    {'TAG meaning:':<15}  ENT   DEP        {'DEP meaning:':<25}")
print (f"{'=====':<15}  {'=====':<15}  {'================':<16}  ====  =========  =======  =====  =====  {'============':<15}  ====  =========  {'==============':<25}")
for tok in ent:
    print (f"{tok.text:<15}  {tok.lemma_:<15}  {str(tok.head):<16}  {tok.i:>4}  {tok.idx:>9}  {tok.idx+len(tok)-1:>7}  {tok.pos_:<5}  {tok.tag_:<5}  {spacy.explain(tok.tag_)[:15]:<15}  {tok.ent_type_:<4}  {tok.dep_:<9}  {spacy.explain(tok.dep_)}  ")
print ("\n")



Token-based analysis of entity: 'Paul Allen':

Token            Lemma            Syntactic parent  #Tok  Chr_Start  Chr_End  POS    TAG    TAG meaning:     ENT   DEP        DEP meaning:             
=====            =====            ================  ====  =========  =======  =====  =====  ============     ====  =========  ==============           
Paul             Paul             fundada              5         29       32  PROPN  PROPN  proper noun      PER   obj        object  
Allen            Allen            Paul                 6         34       38  PROPN  PROPN  proper noun      PER   flat       flat multiword expression  




## 3.2 Visualització d'entitats

spaCy permet representar visualment el text i les seves entitats amb la classe 'displacy':

In [ ]:
colors  = {"ORG": "yellow", "PER": "orange", "LOC": "springgreen", "MISC": "lightgray"}
options = {"colors": colors}
print ("\n")
displacy.render(doc, jupyter=True, style="ent", options = options)
print ("\n")

## 3.3 Model `ca_core_news_trf`. Obtenció entitats

Mateix procés que s'ha seguit per a `ca_core_news_md`, però amb el model `ca_core_news_trf`.

In [ ]:
# Obtenció de les entitats i principals atributs associats d'un text.
text = """Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril de 1975 a Califòrnia per desenvolupar i \
comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat en el processador Intel 8080."""
doc  = get_ents_to_print(nlp_trf, text)


Model 'ca_core_news_trf' applied to text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril de 1975 a Califòrnia per desenvolu
1    par i comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basa
2    t en el processador Intel 8080.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

has detected the entities:

Entity (ent.text)               Type  Tok_Start  Tok_End  Chr_Start  Chr_End  Entities (text string)          Entities (list Token)  
=================               ====  =========  =======  =========  =======  ======================          =====================   
Microsoft                       ORG           0        1          0        9  Microsoft                       [Microsoft]
Paul Allen                      PER           5        7         29       3

El resultat obtingut amb el model `ca_core_news_trf` es diferencia de l'obtingut amb el `ca_core_news_md`:
* Detecta correctament BASIC com una entitat 'MISC' (el model 'md' prediu 'ORG')
* Però detecta erròniament 'Intel 8080' com a 'ORG' (el model 'md' sí detectava 'MISC')

L'apartat [Avaluació dels models NER](#avaluacio) mostra un possible procediment d'avaluació dels models spaCy en català, tot i que amb un fitxer de test molt reduït. S'hi observa que el model `ca_core_news_trf` és el que presenta uns millors resultats.

## 3.4 Representació IOB

L'etiquetatge de seqüències, siguin POS (Part-Of-Speech), chunks (com a sintagmes nominals), entitats anomenades (Named Entities) o altres tipus d'objectes (com per exemple ingredients de receptes, habilitats informàtiques, productes, etc.) es fa identificant si un token és a dins, fora o al principi d'un *objecte*, és a dir Inside, Outside or Beginning, és per això que es coneix com a notació [IOB](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging)).


In [ ]:
text = """Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril de 1975 a Califòrnia per desenvolupar i \
comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat en el processador Intel 8080."""

for token in doc:
    print(token.text, token.ent_type_,token.ent_iob_)

Microsoft ORG B
va  O
ser  O
fundada  O
per  O
Paul PER B
Allen PER I
i  O
Bill PER B
Gates PER I
el  O
4  O
d'  O
abril  O
de  O
1975  O
a  O
Califòrnia LOC B
per  O
desenvolupar  O
i  O
comercialitzar  O
intèrprets  O
de  O
BASIC MISC B
p  O
el  O
Altair MISC B
8800 MISC I
,  O
un  O
microordinador  O
dissenyat  O
el  O
1974  O
i  O
basat  O
en  O
el  O
processador  O
Intel ORG B
8080 ORG I
.  O


## 3.5 Exercicis

1. Provar [les variacions de l'exercici 2.2.1](#exercicis_2_2) amb el model transformer. Milloren els resultats?
2. S'ha vist que els models en català detecten 4 entitats: "PER", "LOC', "ORG" i "MISC". Mirar els tipus d'entitats que es poden detectar amb diferents models de l'anglès.
3. Provar a descarregar un model i fer NER amb Spacy d'un altre idioma.


# <a name="ner_pattern"></a>4 Detectar un nou tipus d'entitat amb patrons

In [ ]:
nlp_patterns = spacy.load("ca_core_news_md")

## <a name="4.1"></a>4.1. Detecció de dates amb patrons (Spacy Matcher)

Els models de català, al contrari de l'anglès, no tenen un tipus d'entitat DATE, encara que detecten algunes dates com del tipus MISC, és a dir com una entitat anomenada d'algun tipus, o 'ORG':

In [ ]:
# Obtenció de les entitats i principals atributs associats al text:
text = """En Pep vindrà el dimecres 23 de setembre del 2020 a Barcelona, o potser el divendres 25. Se n'anirà el 30 de novembre. 
Un altre format: arribarà el 23/9/20 i se n'anirà el 30-11-20, que és el mateix que el 30.11.20, o el 30/11/20."""
doc  = get_ents_to_print(nlp_patterns, text)


Model 'ca_core_news_md' applied to text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    En Pep vindrà el dimecres 23 de setembre del 2020 a Barcelona, o potser el divendres 25. Se n'anirà el 
1    30 de novembre.  Un altre format: arribarà el 23/9/20 i se n'anirà el 30-11-20, que és el mateix que el
2     30.11.20, o el 30/11/20.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

has detected the entities:

Entity (ent.text)               Type  Tok_Start  Tok_End  Chr_Start  Chr_End  Entities (text string)          Entities (list Token)  
=================               ====  =========  =======  =========  =======  ======================          =====================   
Pep                             PER           0        1          0        3  Pep                             [Pep]
Dimecres 23 de setembre         MISC          3        7         14       37  Dime

Es poden observar també els tokens del text:

In [ ]:
tokens_list = list(nlp_patterns(text))
tokens      = "\n".join([token.text for token in tokens_list])  
print(tokens)

Pep
vindrà
el
Dimecres
23
de
Setembre
d
el
2020
a
Barcelona
,
o
potser
el
divendres
25
.
Se
n'
anirà
el
30
de
novembre
.


Altre
format
:
arribarà
el
23/9/20
i
se
n'
anirà
el
30
-
11
-
20
,
que
és
el
mateix
que
el
30.11.20
,
o
el
30/11/20
.


I veure que mentre 23/9/20 o 30/11/20 o 30.11.20 s'analitzen com un únic token, "30-11-20" es divideix en 5 tokens.

A continuació, es detecten les dates en textos de català amb regles basades en patrons (*pattern matching rules*). Per a això, es farà servir el spaCy Matcher. La  documentació és [aquí](https://spacy.io/usage/rule-based-matching#matcher), a la secció **token-based matching**.

 <a name="patterns"></a>

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp_patterns.vocab)

# Dimecres 23 de setembre del 2020
# Dimecres 23 de setembre
# Dimecres 25
pattern1 = [{"LOWER":{"REGEX":r"(dilluns|dimarts|dimecres|dijous|divendres|dissabte|diumenge)"}},
          {"ORTH":{"REGEX": r"([1-9]|[12][0-9]|3[0-1])"}},
           {"ORTH":"de","OP":"?"},
           {"LOWER":{"IN":["gener","febrer","març","marc","abril","maig","juny","juliol","agost","setembre","octubre","novembre","desembre"]}, "OP": "?"},
           {"ORTH":{"REGEX": r"del?"},"OP":"?"},
            {"SHAPE": "dddd", "OP": "?"}
          ]

# 23 de setembre del 2020
# 23 de setembre 2020
# 23 Setembre 2020
# 23 de setembre
# 23 Setembre
pattern2 = [{"LOWER":{"REGEX":r"(dilluns|dimarts|dimecres|dijous|divendres|dissabte|diumenge)"}, "OP": "?"},
          {"ORTH":{"REGEX": r"([1-9]|[12][0-9]|3[0-1])"}},
           {"ORTH":"de","OP":"?"},
           {"LOWER":{"IN":["gener","febrer","març","marc","abril","maig","juny","juliol","agost","setembre","octubre","novembre","desembre"]}},
           {"ORTH":{"REGEX": r"(del?|d')"},"OP":"?"},
            {"SHAPE": "dddd", "OP": "?"}
          ]

# 23/09/2020
# 23-09-2020
# 23.09.2020
# 23/9/20
pattern3 = [ {"ORTH": {"REGEX":r"([0-9]|[12][0-9]|3[0-1])[\/\-\.](0?[1-9]|1[0-2])[\/\-\.][0-9][0-9]([0-9][0-9])?"}} 
           ]

# 23/09/2020
# 23-09-2020
# 23.09.2020
# 23/9/20
pattern4 = [  {"ORTH": {"REGEX":r"([0-9]|[12][0-9]|3[0-1])"}}, # day
              {"ORTH": {"REGEX": r"[\/\-\.]"}} , # separator
              {"ORTH": {"REGEX": r"(0?[1-9]|1[0-2])"}},  # month
              {"ORTH": {"REGEX": r"[\/\-\.]"}} , # separator
              {"ORTH": {"REGEX": r"[0-9][0-9]([0-9][0-9])?"}}  # separator
           ]

matcher.add("DATE", [pattern1, pattern2, pattern3, pattern4], greedy='LONGEST')

print(f"{get_text_to_print(text)}\n")
doc     = nlp_patterns(text)
matches = matcher(doc)
for match_id, start, end in matches:
    string_id = nlp_patterns.vocab.strings[match_id]  # Get string representation
    span      = doc[start:end]  # The matched span
    print(f"{match_id}\t{string_id}\t{start}\t{end}\t{span.text}")

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    En Pep vindrà el Dimecres 23 de setembre del 2020 a Barcelona, o potser el divendres 25. Se n'anirà el 
1    30 de novembre.  Un altre format: arribarà el 23/9/20 i se n'anirà el 30-11-20, que és el mateix que el
2     30.11.20, o el 30/11/20.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

391	DATE	39	44	30-11-20
391	DATE	3	7	Dimecres 23 de setembre
391	DATE	23	26	30 de novembre
391	DATE	16	18	divendres 25
391	DATE	33	34	23/9/20
391	DATE	51	52	30.11.20
391	DATE	55	56	30/11/20


Es pot observar:
 
- Un patró és una seqüència de tokens consecutius, cadascun dels quals es representa com un diccionari. Per tant, tenint en compte que el 23/9/20 es detecta com un token, s’especifica amb una única coincidència de token. En canvi, 23-11-20 es detecta com 5 tokens diferents i per tant necessita una seqüència de 5 tokens.
- "ORTH" cerca una coincidència (‘match’) de cadena exacta, mentre que "LOWER" significa que s’està buscant una correspondència (‘matching’) que no distingeix entre majúscules i minúscules. 
- "OP" significa que el token és opcional. 
- "SHAPE" amb "dddd" significa una seqüència de 4 dígits (d = dígit, en el llenguatge de les expressions regulars) 
- "REGEX" es pot aplicar a qualsevol token i permet cercar una coincidència (‘match’) amb una expressió regular. 
- "IN" cerca una coincidència amb qualsevol valor de la llista.
- greedy= "LONGEST" significa que només es retornen les coincidències més llargues. 


## <a name="4.2"></a>4.2 Incorporació de les noves entitats al pipeline (Spacy EntityRuler)

Per a afegir la detecció de dates amb patrons a les altres entitats detectades amb el model NER, cal agregar un component de tipus [entity ruler](https://spacy.io/usage/rule-based-matching#entityruler) al pipeline *abans* del component NER:

In [ ]:
# primer creem un model des de zero, ja que li afegirem la identificació de dates amb patrons
nlp_patterns = spacy.load("ca_core_news_md")

if "entity_ruler" not in nlp_patterns.pipe_names:
  print("Sequence of tasks in the pipeline *before* adding entity ruler:", nlp_patterns.pipe_names)
  ruler = nlp_patterns.add_pipe("entity_ruler", before="ner")
  print("Sequence of tasks in the pipeline *after* adding entity ruler:", nlp_patterns.pipe_names)
else:
    print("Sequence of tasks in the pipeline:", nlp_patterns.pipe_names)

ruler.add_patterns([{"label":"DATE","pattern":pattern1}, {"label":"DATE","pattern":pattern2},{"label":"DATE","pattern":pattern3},{"label":"DATE","pattern":pattern4}])

print(f"\n{get_text_to_print(text)}\n")
doc = nlp_patterns(text)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Sequence of tasks in the pipeline *before* adding entity ruler: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Sequence of tasks in the pipeline *after* adding entity ruler: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    En Pep vindrà el dimecres 23 de setembre del 2020 a Barcelona, o potser el divendres 25. Se n'anirà el 
1    30 de novembre.  Un altre format: arribarà el 23/9/20 i se n'anirà el 30-11-20, que és el mateix que el
2     30.11.20, o el 30/11/20.
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

Pep 0 3 PER
Dimecres 23 de setembre 14 37 DATE
Barcelona 49 58 LOC
divendres 25 72 84 DATE
30 de novembre 100 114 DATE
23/9/20 143 150 DATE
30-11-20 167 175 DATE
30.11.20 201 209 DATE
30/11/20 216 224 DATE


S'han afegit correctament les entitats anomenades de tipus DATE a altres entitats detectades pel model NER.

## 4.3 Exercicis
 
1. A [4.1](#4.1), provar d'executar la detecció de dates amb Spacy Matcher però sense l'argument `greedy='LONGEST'`. ¿Per a què serveix aquest argument?
 
2. A [4.1](#4.1), provar d'executar cadascun dels 4 patrons per separat per entendre què fan.
 
3. Afegir un altre patró (o modificar-ne un) per tenir en compte un altre format de data.
 
4. A [4.2](#4.2), amb l'entity_ruler, l'argument `before =" ner "` especifica que la detecció de dates amb patrons s'ha d'aplicar abans del component NER. Provar d'eliminar aquest argument `before =" ner "`. Què ha passat i per què?

# <a name="ner_training"></a>5 Afegir un nou tipus d'entitat al model NER


Detectar dates amb patrons és factible, però és una solució "fràgil". Les regles basades en patrons han de ser elaborades i mantingudes manualment, fins i tot en un cas aparentment tan senzill com les dates. 

Una altra solució és entrenar el model NER del català per a reconèixer entitats de tipus DATE. Per a fer-ho, es necessita un conjunt d'oracions anotades amb dates. La secció 5.1 explica com s'ha aconseguit aquest conjunt, fent servir les anotacions obtingudes amb patrons. 

## 5.1 Dades per a l'entrenament

S'han aconseguit les dades per a l'entrenament seguint els passos següents:

1. Preanotació d'un petit conjunt d'oracions amb l'anotació basada en [patrons](#patterns).
2. Correcció de les anotacions en la plataforma d'anotació [INCEpTION](https://inception-project.github.io/).
3. Conversió de les anotacions exportades d'INCEpTION al format CONLL BIO.
4. Conversió de les anotacions en format CONLL BIO al format binari requerit per l'entrenament en Spacy.

Tots aquests passos s'expliquen amb detall en el fitxer anomenat "data_preparation.txt" del apartat de resources/annotation_inception.

Els dos fitxers binaris que es fan servir durant l'entrenament (anomenats training.spacy i dev.spacy) es troben en el directori resources/train_ner_dates. 

Càrrega del fitxer d'entrenament i visualització d'algunes oracions anotades amb Displacy:

In [ ]:
from spacy.tokens import DocBin
import random
doc_bin = DocBin().from_disk("./train_ner_dates/training.spacy")

docs = list(doc_bin.get_docs(nlp_md.vocab))


for i, doc in enumerate(random.sample(docs,5)):
  print(f"\n## Sentence {i+1}:")
  spacy.displacy.render(doc, style="ent", jupyter=True) 


## Sentence 1:



## Sentence 2:


/usr/local/lib/python3.7/dist-packages/spacy/displacy/__init__.py:192: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)



## Sentence 3:



## Sentence 4:



## Sentence 5:


És important observar que aquest corpus d'entrenament no només conté anotacions amb la nova etiqueta DATE, sinó també anotacions amb les altres etiquetes existents del model com PER, ORG o MISC. De no ser així el model NER, que s'hauria entrenat amb anotacions de DATE, però sense anotacions d'etiquetes existents, hauria restat pes a les etiquetes antigues i potser les noves prediccions ometrien aquestes etiquetes. Aquest fenomen s'anomena "catastrophic forgetting". Més informació [aquí](https://explosion.ai/blog/pseudo-rehearsal-catastrophic-forgetting).

## 5.2 Entrenament i predicció

L'entrenament amb Spacy 3 es fa des del 'command line'. Es pot generar automàticament un fitxer de configuració bàsic a partir d'[aquesta pàgina](https://spacy.io/usage/training#quickstart). Aquesta configuració s'ha de completar executant: `python -m spacy init fill-config base_config.cfg config.cfg`

En el fitxer config.cfg del apartat resources/train_ner_dates es pot veure la configuració per a l'entrenament del model de NER del català on s'hi especifica una CPU optimitzada per a l'eficiència. En aquest fitxer es defineixen els paràmetres de l'entrenament, com la llengua, l'ús o no d'un GPU, el tipus d'algoritme d'optimització, el batch size, learning rate, paràmetres de regularització, etc. 

Com que es parteix d'un model existent al qual es vol afegir una nova etiqueta, s'ha hagut de modificar aquesta configuració, especificant que només es vol entrenar el component NER al pipeline, i indicant amb `vectors=ca_core_news_md` que es fan servir els vectors de tokens del model `ca_core_news_md` i que la font del component NER és el model `ca_core_news_md` (`source=ca_core_news_md`).

Un cop es disposa del fitxer de configuració i les dades en format binari, es llança l'entrenament així:

In [ ]:
!python -m spacy train ./train_ner_dates/config.cfg --output ../tmp/output --paths.train ./train_ner_dates/training.spacy --paths.dev ./train_ner_dates/dev.spacy

ℹ Saving to output directory: ../tmp/output
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
[2021-12-03 14:18:46,521] [INFO] Set up nlp object from config
[2021-12-03 14:18:46,534] [INFO] Pipeline: ['ner']
[2021-12-03 14:18:46,534] [INFO] Resuming training for: ['ner']
[2021-12-03 14:18:46,546] [INFO] Created vocabulary
[2021-12-03 14:18:50,425] [INFO] Added vectors: ca_core_news_md
[2021-12-03 14:18:50,804] [INFO] Finished initializing nlp object
[2021-12-03 14:18:50,804] [INFO] Initialized pipeline components: []
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['ner']
ℹ Initial learn rate: 0.001
E    #       LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  --------  ------  ------  ------  ------
  0       0      8.76   68.13   83.04   57.76    0.68
  2     200   1118.65   88.05   89.17   86.96    0.88
  5     400    396.02

- E és el nombre d'èpoques
- \# és el nombre de batches processats
- F, P i R són el F-score, Precisió i Recall, calculats amb el conjunt de desenvolupament.

El millor model, amb el 'score' més alt, es desa al repositori `model-best` dintre del repositori de sortida i l'últim model al repositori `model-last`. 

Amb el model ja entrenat, es poden realitzar prediccions:

In [ ]:
nlp_predict = spacy.load("../tmp/output/model-best") #load the best model
text = """En Pep vindrà el dimecres 23 de setembre del 2020 a Barcelona, o potser el divendres 25. Se n'anirà el 30 de novembre. 
Un altre format: arribarà el 23/9/20 i se n'anirà el 30-11-20, que és el mateix que el 30.11.20, o el 30/11/20."""
doc = nlp_predict(text)
spacy.displacy.render(doc, style="ent", jupyter=True) # display in Jupyter

In [ ]:
text = """Segons l'article de llei 17.50.12 establert a Barcelona 20/12/2001, el pressupost es distribuirà amb proporcions 50-20-30."""
doc = nlp_predict(text)
spacy.displacy.render(doc, style="ent", jupyter=True) # display in Jupyter

## 5.3 Exercicis

1. Comparar les prediccions dels exemples amb les prediccions del model amb dates basat en patrons. Són millors les prediccions?

2. Provar d'entrenar amb GPU el model amb transformers, fent servir el fitxer de configuració anomenat "resources/train_ner_dates/config_trf.cfg"i que trobareu a l'apartat de recursos. Comparar els resultats amb l'altre model, provar de fer prediccions i comparar els resultats.

3. Mirar les anotacions de les dates del fitxer "resources/train_ner_dates/config_trf.cfg"  amb InCEpTION. Us semblen justificades les anotacions? 

4. Donar un cop d'ull a les anotacions [CONLL'2003] (les trobaràs al fitxer anomenat training.conll al directori resources/train_ner_dates. Notar que cada oració està separada amb una línia buida.

# <a name="avaluacio"></a>6 Avaluació dels models NER

L'avaluació d'un model NER es fa típicament comparant les anotacions manuals d'un conjunt de textos (l'anotació de referència o fitxer de test o "ground truth") amb les anotacions del mateix conjunt amb el model NER.

En aquesta secció s'exposa el següent:

* Les principals mètriques associades a NER.
* El procés seguit per aconseguir les anotacions de referència.
* Com aparellar anotacions de referència amb prediccions per fer l'avaluació quantitativa.
* Com calcular les mesures d'avaluació de NER a partir del dataset aparellat.
<p>&nbsp</p>


## 6.1 Mètriques d'avaluació

<a name="Criteris"></a>Mètriques associades amb l'avaluació de models NER: **precisió**, **recall** i **F-score**, el significat de les quals és l'usual estadístic, calculades a partir del recompte dels casos TP, FP i FN (veritables positius, falsos positius i falsos negatius, respectivament).  

Ara bé, una entitat consta de 'Tipus d'entitat' i 'Fronteres d'entitat' ('tokens' inicial i final que delimiten l'entitat). D'acord amb aquests dos atributs, 'Tipus' i 'Fronteres', el recompte de TP, FP i FN és diferent en funció de diferents criteris d'avaluació que es poden establir:
* 'ESTRICTE': Coincidència de tipus i fronteres.
* 'EXACTE': Coincidència de fronteres amb independència de tipus.
* 'PARCIAL': Coincidència parcial de fronteres amb independència de tipus.
* 'TIPUS": Només coincidència de tipus.

Documents com ara [Evaluation of the SemeVal-2013 Task 9.1](https://www.cs.york.ac.uk/semeval-2013/task9/data/uploads/semeval_2013-task-9_1-evaluation-metrics.pdf) o [nervaluate Project description](https://pypi.org/project/nervaluate/#description) descriuen cada criteri d'avaluació i, en funció del criteri, com realitzar el recompte de TP, FP i FN. 

<p>&nbsp</p>

**Avaluació dels models spaCy per a català:** d'acord amb la [documentació dels models](https://spacy.io/models/ca) (apartat 'Accuracy evaluation' de cada model), les magnituds **precisió**, **recall** i **F-score** són:

Model |Precisió | Recall | F-Score
------|:-------:|:------:|:-------:
ca_core_news_sm|80%|80%|80%
ca_core_news_md|84%|84%|84%
ca_core_news_lg|85%|84%|85%
ca_core_news_trf|92%|91%|91%

on el criteri utilitzat per a avaluar ha estat 'ESTRICTE'.


## 6.2 Anotació de referència

L'obtenció de textos anotats per a l'avaluació de models NER és una etapa crítica en l'avaluació de tasques NLP, tot i que tediosa i sovint amb un elevat consum de recursos i temps. El procés d'anotació del "ground truth" és semblant al que s'ha seguit per a aconseguir el dataset d'entrenament, encara que aquesta vegada s'ha fet servir l'eina d'anotació en línia [m47.ai](https://www.m47.ai/ca/):

1. S'ha partit d'un conjunt de 10 paràgrafs extrets de Viquipèdia (el csv el trobareu el fitxer UOC_NER_Textos_per_anotar del directori resources/annotation_m47.

2. Amb m47, s'han preanotat manualment els textos usant el model per a català 'Aina NER (based on spaCy)' [3] ja desplegat a l'eina. 

3. S'han revisat les anotacions manualment.

4. S'han exportat les anotacions a un format json, s'han convertit a anotacions Spacy i desat a un fitxer pickle.

Aquest procés s'explica amb més detall en el fitxer m47.txt del apartat resources/annotation_m47.

[3] Versió preliminar de ca_core_news_lg (https://github.com/TeMU-BSC/spacy/releases/tag/v3.2.4lg), a la data d'anotació del fitxer anotat per a aquest Notebook: octubre 2021.

Càrrega de les anotacions de referència des del fitxer 'pickle' i impressió d'una anotació com a exemple. El format de l'anotació es una tupla amb el text com a primer element i una llista d'entitats com a segon element.

In [ ]:
import pickle

fh = open('annotation_m47/m47_ner.pkl', 'rb') 
wiki_annotations = pickle.load(fh)
wiki_annotations[0]

('Einstein va néixer a Ulm (Baden-Württemberg) el 14 de març del 1879. Va créixer a Munic i, més tard, a Itàlia.\nMalgrat que arribaria a ser un dels més importants físics teòrics, un mestre seu va dir al seu pare: "No farà mai res de profit". Interessat en les matemàtiques als dotze anys, als quinze es va sentir atret per l\'àlgebra i la geometria i, finalment, pel càlcul infinitesimal. Tot i haver-se graduat a l\'Escola Politècnica Federal de Zuric l\'any 1900 com a professor de matemàtiques i física, no va poder tenir cap plaça a la universitat. El 1896, renuncià a la ciutadania alemanya, i el 1901 va obtenir la nacionalitat suïssa. Sense poder accedir a la universitat, cercà una feina temporal a Berna i, posteriorment, una d\'indefinida a l\'Oficina de Patents suïsses, el 1904.\nEl 1895, als setze anys, escriu el seu primer assaig científic: Sobre la investigació de l\'estat de l\'èter en un camp magnètic. Seguí la formació superior a Suïssa, a l\'ETHZ, gràcies al fet que el seu di

## 6.3 Aparellament de les prediccions i anotacions manuals (Spacy `Example`)

Donat el conjunt d'anotacions de referència, es pot fer la predicció de cada text amb el model NER i aparellar les dues anotacions amb l'objecte de Spacy [Exemple](https://spacy.io/api/example). Aquest objecte és la base per fer l'avaluació quantitativa, ja que cal comparar les dues anotacions. Aquí es defineix la funció que fa aquest aparellament:

In [ ]:
def get_examples(model=None, annotations=None):
  """Obtain 'example' spaCy objects (text with predict and gold reference annotations) from a list of annotated text (gold reference or ground-truth).

    Parameters:
      model (spaCy model): spaCy model for entity recognition.
      annotations (list): list of tuples. Every tuple 'k' of the list is a text with its gold reference entities formatted as:
        annotations[k][0] (str): text
        annotations[k][1] (list): gold reference list of tuples, each tuple has 3 elements:
          -First (int): Chr. start of entity
          -Segon (int): Chr. end + 1 of entity
          -Third (str): Entity type

    Returns:
      list: List of 'example' spaCy objects, one 'example' for every text in 'annotations' parameter. 
    """
  if model is None or annotations is None:
    print("Function 'get_examples' needs model and annotations")
    return []
    
  ex_docs = []
  # Obtenir llista d'examples spaCy per a avaluar posteriorment
  for k, (text, annot) in enumerate(annotations):
      doc = model(text)                                   # Predicció del model

      # Construir objecte Example. Conté la predicció i l'anotació manual (https://spacy.io/api/example#from_dict)
      ex  = Example.from_dict(doc, {"entities": annot})   
      
      # Acumular tots els Example de cada element del corpus.
      ex_docs.append(ex)
  return ex_docs

Funció per imprimir el text, la seva predicció i anotació manual:

In [ ]:
def print_example_entities_from (spacy_doc):
  """Print entities of a 'doc' spaCy.

    Parameters:
     spacy_doc: spaCy 'doc'

    Returns: ---
  """
  tokens_list = list(spacy_doc)    # Cada element de la llista és un objecte Token de spaCy
  text        = spacy_doc.text     # text corresponent a ex.reference
  print (f"{'Entity (ent.text)':<30}  Type  Tok_Start  Tok_End  Chr_Start  Chr_End  {'Entities (text string)':<30}  {'Entities (list Token)'}  ")
  print (f"{'=================':<30}  ====  =========  =======  =========  =======  {'======================':<30}  {'====================='}   ")
  for ent in spacy_doc.ents:
      print (f"{ent.text:<30}  {ent.label_:<4}  {ent.start:>9}  {ent.end:>7}  {ent.start_char:>9}  {ent.end_char:>7}  {text[ent.start_char:ent.end_char]:<30}  {tokens_list[ent.start:ent.end]}")
  print ("\n") 

In [ ]:
def print_example(model, ex: Example, do_print_text=True, do_print_prediction=True, do_print_ground_truth=True):
  """Print text and predicted and gold reference entities from an 'example' spaCy object.

    Parameters:
      model (spaCy model): model for entity recognition
      ex ('example' spaCy object): contains text and predicted and gold reference entities annotations
      do_print_text (bool): true when print text required
      do_print_prediction (bool): true when print entities predicted required
      do_print_ground_truth (bool): true when print gold entities required

    Returns: --
  """
  
  if do_print_text:
    print ("\nEntity recognition of the text:")
    print(f"\n{get_text_to_print(ex.predicted.text)}")

  if do_print_ground_truth:
    print (f"\nGOLD REFERENCE entities:\n")    
    print_example_entities_from (ex.reference) 

  if do_print_prediction:
    print (f"\nPREDICTED entities with the model '{model.meta['lang']+'_'+model.meta['name']}':\n")      
    print_example_entities_from (ex.predicted)
  

In [ ]:
wiki_examples = get_examples(nlp_md, wiki_annotations)
print_example(nlp_md, wiki_examples[2])


Entity recognition of the text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    Google Inc. és l'empresa propietària de la marca Google, el producte principal de la qual és el moto
1    r de recerca del mateix nom. Va ser fundada el 4 de setembre del 1998 per Larry Page i Sergey Brin, 
2    dos estudiants de doctorat en ciències de la computació de la Universitat de Stanford, que el gener 
3    del 1996 van aconseguir un cercador més eficaç anomenat Google, capaç de mostrar els resultats de la 
4    recerca en un ordre jeràrquic condicionat pel nombre de visites
     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100

GOLD REFERENCE entities:

Entity (ent.text)               Type  Tok_Start  Tok_End  Chr_Start  Chr_End  Entities (text string)          Entities (list Token)  
=================               ====  =========  =======  =========  =======  =====================

## 6.4 Avaluació dels models

Aquesta secció avalua NER utilitzant dues llibreries diferents:

1.   Amb spaCy, classe 'scorer'. Avalua segons criteri 'ESTRICTE', tal com s'ha definit a [6.1](#Criteris), i per al conjunt dels tipus d'entitats anotades.
2.   Amb nervaluate. Avalua segons tots els criteris definits a [6.1](#Criteris) i per al conjunt dels tipus d'entitats, però també per cada tipus d'entitat per separat (consegüentment, les mètriques del criteri 'strict' de nervaluate i tipus entitat 'TOTAL' coincideixen amb les de spacy.scorer)


### 6.4.1 Avaluar amb spaCy `Scorer`

Senzillament es crida el mètode [score_spans](https://spacy.io/api/scorer#score_spans) d'un objecte de tipus [Scorer](https://spacy.io/api/scorer), amb la llista d'objectes de tipus [Example](https://spacy.io/api/example) com a argument d'entrada:

In [ ]:
def score_ner(examples):
  """Evaluate spaCy model with the spaCy Scorer class.

    Parameters:
      examples (list): list of 'example' objects spaCy: documents spaCy annotated (predicted and gold reference).

    Returns: 
      dicc: result of the evaluation. See diccionary format at: https://spacy.io/api/scorer#score_spans   
  """
  scorer  = Scorer()
  # Avaluar 'examples' i retornar resultat de l'avaluació  
  return scorer.score_spans(examples, "ents")      # https://spacy.io/api/scorer#score_spans


In [ ]:
def print_metrics (metrics, model):
  """Print results of the spaCy entity evaluation.

    Parameters:
      metrics (dicc): results of the spaCy entity evaluation; dicc with a spaCy format (see paragraph 'returns' in https://spacy.io/api/scorer#score_spans)
      model (spaCy model): spaCy model for NER

    Returns: --- 
  """
  df = pd.DataFrame([metrics["ents_p"], metrics["ents_r"], metrics["ents_f"]], columns=["Total"], index=["Precisió", "Recall", "F-score"] )
  for items in metrics["ents_per_type"].items():
    df[items[0]] = [items[1]["p"], items[1]["r"], items[1]["f"] ]
  
  df = (df*100).round(decimals = 2)

  print (f"\nEvaluation of the model: '{model.meta['lang']+'_'+model.meta['name']}'")
  print ("===============================================\n")
  print (df)
  print ("\n")


Donat un model i uns textos anotats amb entitats al format Spacy, es crea la llista d'objectes de tipus Exemple, s'avalua i s'imprimeix:

In [ ]:
def ner_evaluation(model=None, annotations=None):
  """Evaluate NER with spaCy. Print the evaluation
  
    Parameters:
      model (spaCy model): spaCy model for NER
      annotations (list): list of tuples. Every tuple 'k' of the list is a text with its gold reference entities formatted as:
        annotations[k][0] (str): text
        annotations[k][1] (list): gold reference list of tuples, each tuple with 3 elements:
          -First (int): Chr. start of entity
          -Second (int): Chr. end + 1 of entity
          -Third (str): Entity type

    Returns: 
      list: list of spaCy 'example' objects (every 'example': text and predicted and gold reference entities)
  """
  if model is None or annotations is None:
    print("ner_evaluation must have valid arguments")
    return []
  
  examples_obj = get_examples(model, annotations)
  metrics = score_ner(examples_obj) 
  print_metrics(metrics, model) 
  return examples_obj

Avaluar els dos models:

In [ ]:
examples_md = ner_evaluation(nlp_md, wiki_annotations)

examples_trf = ner_evaluation(nlp_trf, wiki_annotations)

2021-12-03 14:16:08 numexpr.utils INFO: NumExpr defaulting to 2 threads.



Evaluation of the model: 'ca_core_news_md'

          Total    ORG    PER    LOC   MISC
Precisió  63.04  62.82  73.33  77.08  36.67
Recall    62.82  52.13  78.57  63.79  53.66
F-score   62.93  56.98  75.86  69.81  43.56



Evaluation of the model: 'ca_core_news_trf'

          Total    ORG   PER    LOC   MISC
Precisió  81.59  77.32  88.1  92.86  62.50
Recall    81.59  79.79  88.1  89.66  60.98
F-score   81.59  78.53  88.1  91.23  61.73




A efectes de comprovació, es poden imprimir les anotacions dels dos models d'un dels textos:

In [ ]:
print_example(nlp_md, examples_md[0], do_print_text=True, do_print_ground_truth=True, do_print_prediction=True)
print_example(nlp_trf, examples_trf[0], do_print_text=False, do_print_ground_truth=False, do_print_prediction=True)


Entity recognition of the text:

     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100
0    Einstein va néixer a Ulm (Baden-Württemberg) el 14 de març del 1879. Va créixer a Munic i, més tard, 
1    a Itàlia. Malgrat que arribaria a ser un dels més importants físics teòrics, un mestre seu va dir al
2     seu pare: "No farà mai res de profit". Interessat en les matemàtiques als dotze anys, als quinze es va sen
3    tir atret per l'àlgebra i la geometria i, finalment, pel càlcul infinitesimal. Tot i haver-se gradua
4    t a l'Escola Politècnica Federal de Zuric l'any 1900 com a professor de matemàtiques i física, no va
5     poder tenir cap plaça a la universitat. El 1896, renuncià a la ciutadania alemanya, i el 1901 va ob
6    tenir la nacionalitat suïssa. Sense poder accedir a la universitat, cercà una feina temporal a Berna
7     i, posteriorment, una d'indefinida a l'Oficina de Patents suïsses, el 1904. El 1895, als setze anys, e
8

### 6.4.2 Avaluar amb nervaluate

La llibreria '[nervaluate](https://pypi.org/project/nervaluate/)' amplia el càlcul de **precisió**, **recall** i **F-score**,  als diferents criteris d'avaluació mencionats a la secció anterior ('[Mètriques d'avaluació](#Criteris)' amb el detall, a més, del recompte de la classificació d'entitats d'acord amb els grups identificats a '[nervaluate](https://pypi.org/project/nervaluate/)'.


In [ ]:
def nervaluate_evaluation(model, examples, labels):
  """Evaluate NER with nervaluate library (https://pypi.org/project/nervaluate/)

    Parameters:
      model (spaCy model): spaCy model for NER
      examples (list): list of tuples. Every tuple 'k' of the list is a text with its gold reference entities formatted as:
        examples[k][0] (str): text
        examples[k][1] (list): gold reference list of tuples, each tuple with 3 elements:
          -First (int): Chr. start of entity
          -Second (int): Chr. end + 1 of entity
          -Third (str): Entity type
      labels (list): list of labels to be evaluated

    Returns: 
      dicc: Metrics of evaluation, 'total'. See diccionay format at: https://pypi.org/project/nervaluate/ 
      dicc: Metrics of evaluation, 'by label'. See diccionay format at: https://pypi.org/project/nervaluate/ 
  """
  label_dir   = {label:index for index, label in enumerate(labels)}   # Passa la llista d'etiquetes a diccionari del tipus: {'PER': 0, 'LOC': 1, 'ORG': 2, 'MISC': 3}
  prediction  = [[] for _ in labels]   # Per a acumular anotacions predites agrupades per les etiquetes. Requerit per nervaluate
  real        = [[] for _ in labels]   # Per a acumular anotacions manuals agrupades per les etiquetes. Requerit per nervaluate
  offset      = 0                      # Donat que s'acumulen totes les entitats de tots els docs, acumular longituds de cada doc.

  # Preparar, en format nervaluate, els textos que el model prediu (llista 'prediction') i l'anotació manual (llista "real")
  # Format nervaluate: veure: https://pypi.org/project/nervaluate/  
  for text, annot in examples:

    # Construir el format nervaluate de la predicció
    doc = model (text)    
    for ent in doc.ents:
      if ent.label_ not in labels:
        print (f"Warning! Type entity predicted unknown. Omitted. Type: '{ent.label_}'. Entity: '{ent.text}'.\nText: {text}")
        continue
      dic = {"label": ent.label_, "start": ent.start_char+offset, "end": ent.end_char+offset}
      
      # Acumular totes les entitats de tots els documents en una única llista de predicció (d'acord amb la documentació nervaluate)
      prediction[label_dir[ent.label_]].append(dic)
    
    # Construir el format nervaluate de l'anotació manual
    for ent in annot:
      if ent[2] not in labels:
        print (f"Warning! Type gold reference unknown. Omitted. Type: '{ent[2]}'. Entity char starts: {ent[0]}, char ends: {ent[1]}.\nText: {text}")
        continue
      dic = {"label": ent[2], "start": ent[0]+offset, "end": ent[1]+offset}
      
      # Acumular totes les entitats de tots els documents en una única llista d'anotació manual (d'acord amb la documentació nervaluate)  
      real[label_dir[ent[2]]].append(dic)

    # Donat que tots els documents s'acumulen en úniques llistes de predicció i anotació manual, anar desplaçant posicions amb les longituds acumulades dels documents
    offset = offset + len(text)

  evaluator = Evaluator(real, prediction, tags=labels)     # avaluar amb nervaluate
  metrics, metrics_by_label = evaluator.evaluate()         # veure https://pypi.org/project/nervaluate/

  return metrics, metrics_by_label



In [ ]:
# Donat un diccionari amb el format d'avaluació de nervaluate, formatar-lo com a df per imprimir-lo
def format_df (dicc):
  """Format a dictionary to be printed"""
  df = pd.DataFrame(dicc)
  df.loc[['precision', 'recall', 'f1']] = (df.loc[['precision', 'recall', 'f1']]*100).round(decimals = 2)
  df = df.astype(object)         
  return df

In [ ]:
def print_nervaluate (metrics, metrics_by_label, model, labels ):
  """Print metrics of the nervaluate evaluation

    Parameters:
      metrics (df): Metrics of evaluation, 'total'. 
      metrics_by_label (df): Metrics of evaluation, 'by label'
      model (spaCy model): spaCy model for NER
      labels (list): list of labels evaluated

    Returns: --- 
  """
  # metriques: diccionari que conté l'avaluació del model pel conjunt de totes les etiquetes
  df_total = format_df (metrics)
  
  print (f"\nEvaluation of the model '{model.meta['lang']+'_'+model.meta['name']}':")
  print ("===============================================\n")
  print (f"Metrics TOTAL:")
  print (df_total)

  # metrics_by_label: diccionari que conté l'avaluació del model per etiqueta
  for label in labels:
      df_label = format_df (metrics_by_label[label])
      print (f"\nMetrics by {label}:")
      print (df_label)

  print ("\n")



In [ ]:
LABELS = ["PER", "LOC", "ORG", "MISC"]     # Nervaluate avalua per etiquetes

metrics, metrics_by_label = nervaluate_evaluation(nlp_md, wiki_annotations, LABELS) 
print_nervaluate (metrics, metrics_by_label, nlp_md, LABELS)

metrics, metrics_by_label = nervaluate_evaluation(nlp_trf, wiki_annotations, LABELS) 
print_nervaluate (metrics, metrics_by_label, nlp_trf, LABELS)




Evaluation of the model 'ca_core_news_md':

Metrics TOTAL:
          ent_type partial strict  exact
correct        195     174    174    174
incorrect        0       0     21     21
partial          0      21      0      0
missed          82      82     82     82
spurious        81      81     81     81
possible       277     277    277    277
actual         276     276    276    276
precision    70.65   66.85  63.04  63.04
recall        70.4   66.61  62.82  62.82
f1           70.52   66.73  62.93  62.93

Metrics by PER:
          ent_type partial strict  exact
correct         73      66     66     66
incorrect        0       0      7      7
partial          0       7      0      0
missed          11      11     11     11
spurious        17      17     17     17
possible        84      84     84     84
actual          90      90     90     90
precision    81.11   77.22  73.33  73.33
recall        86.9   82.74  78.57  78.57
f1           83.91   79.89  75.86  75.86

Metrics by LOC:
    


Donat que aquest notebook només pretén presentar una prova de concepte al voltant de diferents aspectes de NER i la limitada dimensió del corpus utilitzat per a avaluar ('wiki_annotations' només disposa de 314 entitats), no es pot establir cap conclusió a partir dels resultats de l'avaluació realitzada, observar només les tendències per al corpus 'wiki_annotations':

* Model `'ca_core_news_md'`:   
F-score segons documentació spaCy: 84%    
F-score amb 'wiki_annotations': 62,93%.   

El resultat és lluny de l'esperat. Una anàlisi d'algun dels exemples mostra la rellevància dels criteris emprats per a l'anotació manual i en alguns casos (registre 8 sobre Galileu) un comportament del model erràtic. El tipus d'entitat 'MISC' mostra els pitjors resultats (F-score 43,56%), probablement per la manca de coneixement dels criteris utilitzats en l'entrenament del model per spaCy i que poden no correspondre als aplicats a l'anotació manual realitzada per a aquest notebook.

* Model `'ca_core_news_trf'`:  
F-score segons documentació spaCy: 91%  
F-score amb 'wiki_annotations': 81,59%  

Els resultats s'acosten als esperats i, encara més, si es té en compte el resultat desglossat per tipus d'entitat (PER: 88,1%; LOC: 91,23%; ORG: 78,53%; MISC: 61,73%). Com en el model anterior, MISC presenta el pitjor resultat, probablement per les mateixes raons ja exposades; d'altra banda, de l'anàlisi d'exemples concrets hi ha sovint un transvasament entre entitats 'ORG' i 'MISC', si no existís aquest transvasament els resultats millorarien el 81,59% observat.

## 6.5 Exercicis

1.   Escollir un text, anotar-lo manualment en format spaCy, procurar que almenys hi hagi tres mencions per a cada tipus d'entitat: 'PER', 'LOC', 'ORG' i 'MISC', algunes d'elles multitoken. Fer la predicció amb el model `ca_core_news_md`. Avaluar el model amb `nervaluate`. Comprovar el resultat de `nervaluate` recomptant manualment els casos TP, FP i FN i calculant precisió, recall i F-score per a cada categoria i tipus d'entitat.  
2. Mateix exercici anterior, però amb el model: `ca_core_news_trf`.
3.   Crear un corpus amb diferents textos per escollir (articles de diari, Viquipèdia, fòrums...) i anotar-los amb una eina de mercat (vegeu secció [Recursos](#recursos)), tipus entitat 'PER', 'LOC', 'ORG', 'MISC' i 'DATE'. Utilitzar la capacitat de preanotació de l'eina si en té. Convertir el resultat de l'anotació a format spaCy. 
4. Amb el corpus anotat de l'exercici anterior, avaluar el model entrenat a la [secció 5](#ner_training) per als cinc tipus d'entitats. Revisar exemples. Analitzar el resultat. Hi ha molts errors de predicció? És un problema de criteris a l'hora d'anotar manualment? És un problema del model? 


# <a name="recursos"></a>7 Recursos

- Hi ha moltes eines d'anotació de textos disponibles: [m47.ai](https:////www.m47.ai/ca/); [INCEpTION](https://inception-project.github.io/), [label studio](https://labelstud.io/), [doccano](https://doccano.herokuapp.com/), [tagtog](https://www.tagtog.net/), [prodigy](https://prodi.gy/) (part de l'univers spaCy, versió de pagament), [brat](https://brat.nlplab.org/index.html), [ubiai](https://ubiai.tools/)...
- Altres llibreries de PLN que fan NER: [Stanza](https://stanfordnlp.github.io/stanza/ner.html), [Stanford CoreNLP](https://stanfordnlp.github.io/CoreNLP/ner.html), [Flair](https://github.com/flairNLP/flair), [SparkNLP](https://nlp.johnsnowlabs.com/docs/en/transformers), [AllenNLP](https://demo.allennlp.org/named-entity-recognition/named-entity-recognition). 
- [SPACY v3: State-of-the-art NLP from Prototype to Production](https://www.youtube.com/watch?v=9k_EfV7Cns0&t=114s) (vídeo)
- [Sequence Models for Named Entity Recognition](https://www.youtube.com/watch?v=46Hfn8dwUEs) (vídeo d'introducció pel Prof. Chris Manning)
- Speech and Language Processing. Daniel Jurafsky & James H. Martin. 2021. Chapter 8 on ["Sequence Labeling for Parts of Speech And Named Entities"](https://web.stanford.edu/~jurafsky/slp3/8.pdf) and Chapter 71 on ["Information Extraction"](https://web.stanford.edu/~jurafsky/slp3/17.pdf).